# <Paper title>: reproduction and extension

**Student:** <name> · **Paper:** <authors, title, venue, year, link> · **Track:** Docker (ns-3 / Python) or NixNet

**Summary (3-5 sentences):** what the paper claims, what you reproduced, what you changed, and the main result.

> **How to use this template** (delete this box before submitting)
> 1. Keep the sections of the track you use; delete the example sections you do not need.
> 2. Replace the example scenario (`ns3/aanet`, `sim/main.py` or the experiment in `flake.nix`) with your own.
> 3. Every experiment runs through `runner.py`, which caches results in `results/`. Re-running a cell only runs what is missing.
> 4. Before submitting: commit, clone into a fresh folder, follow the README and run this notebook top to bottom (*Kernel → Restart & Run All*).

## 0. Related Work/Setting
A Unmanned aerial vehicle (UAV) swarm is a group of drones that work together in a coordinated, autonomous way. In particular, they must maintain accruate and synchronized state information (position, velocity, heading) even in the face of partial failures and environmental interference. (UAV swarms rely heavily on precise coordination, among others, of their positions). To this effect, UAVs are equipped both with Global Navigation Satellite Systems (GNSS) and Inertial Navigation Systems (INS). GNSS signals can be unreliable in real-world conditions, due to dense vegetation, urban canyons or subterranean environments. In addition, GNSS technologies are vulnerable to spoofing or jamming. So, in attempts to make up for these short-comings, UAVs are also equipped with INS systems. These use accelerometer and gyroscope information to estimate the current position of an UAV, given a starting point. One could think of this as moving through a room with the eyes closed. One is at the door, counts how many steps one takes and how often one turns, and hopefully one would have a rough estimate of where one is in the room. Because INS does not rely on external radio signals, it is harder to spoof and can be used in areas where GNSS fails. However, it is less accurate and the accumulation of small errors in pose estimation can cause the estimated position to drift away from its correct one. The limitations of GNSS and INS as standalone systems speak for the use of collaborative, consensus-driven approaches, that combine GNSS, INS and leverage the connections between UAVs in a swarm as a way to achieve a robust and fault-tolerant state estimation for UAV swarms.<br>
System Model: <br>
For this particular paper, we assume the communication graph among the UAVs to be fully connected and synchronous. So every UAV can has a direct link to every other UAV and we assume that the network conditions are such that we don't need to worry about a message that was sent in time step k only being received on time step k+1. The paper does this so it can focus on sensor faults instead of network delays. We also assume that the UAV carries a ranging sensor of the Received Signal Strength Indicator (RSSI) type. This sensor measures the strength of a received signal and by applying a signal propagation model allows a noisy estimation of distance. A lower received signal means the sender is farther away, whilst a higher means it is closer. Walls, obstacles, antenna orientation and the choice of the propagation model all influence the computed distance, so the result is always noisy. RSSI is used in the paper as a way for the UAV's to have some notion of where all other nodes are (also called internode distances). <br>
Messages between nodes are sent over an authenticated broadcast channel protected by pre-distributed cryptographic keys, so message forgery or tampering are detected. <br>
INS can always be relied upon. Only GNSS sensors can be Byzantine (i.e. up to f can fail)


# Operating Conditions:
A UAV swarm is a speciallized case of a distrbitued system, but it can in some ways still be characterized as one. Swarm consensus should: 
* have low-latency agreements
* operate under strict ressource constraints (limited compute/energy resources; intermittent connectivity) / be lightweight
* be robust agains partial system failure
* ensure termination (all non-faulty agents reach a decision)
* ensure agreement (decisions are consistent)
* ensure integrity (decisions are based on valid inputs)

## 1. High Level Idea
The paper's key idea was to recognize the commonalities between a swarm and a distributed system, and then adapt a well-established consensus algorithm for distributed systems to our scenario. The used protocol is Raft, usually used for coordination of data storage in databases. More importantly, Raft has a lightweithgt leader architecture and is crash-tolerant. It can recover from network partitions, where some nodes get disconnected from the network for some time and upon reconnecting have to synchronize their storage. The paper adapts the Raft algorithm to enable drones to communicated and synchronize their positions over a network, even in partially GNSS denied environments. When GNSS fails, the swarm uses consensus to reconstruct/verify the location and trajectory of affected nodes based on data they shared and how the affected drones where moving prior to failing. In this way, even when individual drones's sensing mechanisms fail, the swarm remains cohesive. The paper's **SwarmRaft** protocol thus makes use of bothe the GNSS and INS data to produce a robust state estimate to reduce the effects of signal failure, spoofing and sensor drift, with it improving the swarm's situational awareness and operational reliability.

## 2. The Workflow
At each time step k, the swarm determines the position of each UAV. To do so, nodes communicate via Raft consensus and elect a leader for that time step. The leader collects every node's position estimate (via GNSS or INS when the other failed) tests to see if the node's reported position is consistent with what the other nodes are seeing, recovers the correct position and sends the results to the nodes. In **SwarmRaft**, so long as n >= 2f + 1 where n is the number of nodes and f the amount of corrupted nodes, the system will be fault tolerant (i.e. if the majority of the nodes is not faulty, the system can recover from incorrectly reported data). <br>
Given a leader for a time step k, the **SwarmRaft** does the following:
1. **SENSE:** Each UAV measures its position (GNSS/INS) and internode distances (each node estimates distance to neighbouring nodes).
2. **INFORM:** Measurements are sent to the leader.
3. **ESTIMATE:** Leader recomputes each UAV's position from range constraints (sanity check).
4. **EVAULUATE:** Leader determines if the received sensor data is faulty or honest based on allowed deviations.
5. **RECOVER:** If sensor data is faulty, that data is replaced with position computed with other drones's estimates.
6. **FINALIZE:** Leader sends final results to all nodes.

## 3. Threat Model
Now we will introduce the attack scenarios that **SwarRaft** is prepared to handle.
1. **GNSS Spoofing Attack:** Attacker is able to distort the position measurements of the GNSS of up to f nodes, causing each node to report a position that is offset by some bias. Our inter-node consistency checks and the consensus mechanism ensure that GNSS readings that deviate significantly from the position estimations of other UAVs get detected.
2. **Ranging-Tampering Attack:** Attacker interferes with the RSSI sensors and is able to create errors in the inter-UAV distance measurements. ?????? (Solution part not clear yet)
3. **Collusion Among Faulty Sensors:** Up to f faulty UAVs decide to collude on sensor information. Compromised nodes coordinate their reported measurements and their votes in the **SwarmRaft** protocol, in hopes of influencing the swarm's decision about the position of a particular "victim" node. Due to the majority requirement, the scheme guarantees that you always have a higher number of honest votes than colluding malicious votes, so no coallition (of size <= f) can force an incorrect decision for the whole swarm.

## 4. The Algorithm in Depth: Pose Estimation
At each time step k $\in$ $\mathbb{N}$ the UAV i $\in$ {1, ..., n} gets two complementary measurements about its state. The GNSS receiver produces a direct, but noisy (see before) readout of the true position x_(i,k): __formula__ . <br>
The INS computes the estimation of the position based on its previous estimation and increments derived from the gyroscopes and accelerometers's behaviour during the time in between step [k-1, k], called the inertial measurements. <br>

In short: INS has high-rate relative motion at the cost of accumulating drift vs GNSS's drift-free absolute position, but with high instant noise. <br>

The information from the two sensors gets cross-checked and each UAV measures its distance to every other UAV. The reading satisfies the formula: __formula__. This just means that the measured distance is the true distance, offset by the measurement noise of the method. This geometric constraint links the true positions of both drones is used to detect inconsistencies in node's measurements (due to GNSS/INS spoofing or drift).

## 5. The Algorithm in Depth: Fault Detection
## 6. The Algorithm in Depth: Position Correction and Recovery


## 0. Setup
`ENGINE="docker"` runs every job in the pinned Docker images and is required for the results you submit.
`ENGINE="local"` runs the same jobs without Docker and is only for debugging.

In [ ]:
import os

os.environ.setdefault("ENGINE", "docker")  # set before importing runner
TRACKS = os.environ.get("TRACKS", "ns3,python,nixnet").split(",")  # delete what you do not use

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import runner
from runner import grid, read_iperf3, read_ping, run_nixnet, run_ns3, run_py, sweep

FIG = Path("figures")
FIG.mkdir(exist_ok=True)


def mean_ci(df, by, col):
    """Mean of `col` over seeds with an approximate 95% confidence interval (1.96 x SEM)."""
    g = df.groupby(by)[col]
    out = g.agg(["mean", "std", "count"])
    out["ci95"] = 1.96 * out["std"] / np.sqrt(out["count"])
    return out.reset_index()


pd.Series(runner.environment_info(), name="value").to_frame()  # report this with your results

## 1. Experiment design
- **Research question:** <what the experiment answers>
- **Parameters swept:** <names, values, why these values>
- **Fixed parameters:** <and where they come from in the paper>
- **Metrics:** <definitions and units>
- **Repetitions:** <number of seeds/runs and why it is enough>

## 2. ns-3 experiment (Docker track)
Example: AANET snapshot (`ns3/aanet/aanet.cc`). Aircraft in a 1000 km x 1000 km area, air-to-air links within `commRange`,
min-hop routing, 10 random UDP flows. Each job is one `(params, seed)` pair; the seed is passed to ns-3 as `--RngRun`.

In [ ]:
if "ns3" in TRACKS:
    ns3_configs = grid(numNodes=[50, 100], commRange=[200_000, 300_000, 400_000], simTime=[10])
    ns3_seeds = range(1, 6)
    ns3_df = sweep(run_ns3, ns3_configs, ns3_seeds)  # one row per flow, per job
    display(ns3_df.head())

In [ ]:
if "ns3" in TRACKS:
    # aggregate flows per run first, then compute mean and CI over seeds
    per_run = (ns3_df.groupby(["numNodes", "commRange", "seed"])
               .agg(reachable=("reachable", "mean"), delay_ms=("mean_delay_ms", "mean"))
               .reset_index())
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for ax, col, label in [(axes[0], "reachable", "Reachable flows (fraction)"),
                           (axes[1], "delay_ms", "Mean end-to-end delay [ms]")]:
        s = mean_ci(per_run, ["numNodes", "commRange"], col)
        for n, g in s.groupby("numNodes"):
            ax.errorbar(g.commRange / 1e3, g["mean"], yerr=g.ci95, marker="o", capsize=3, label=f"{n} aircraft")
        ax.set_xlabel("Air-to-air range [km]")
        ax.set_ylabel(label)
        ax.grid(alpha=0.3)
    axes[0].legend()
    fig.tight_layout()
    fig.savefig(FIG / "ns3_reachability_delay.pdf", bbox_inches="tight")

## 3. Python simulation (Docker track)
Example: Monte Carlo of fleet connectivity to two ground stations (`sim/main.py`). The script receives its parameters as JSON
and the seed on the command line, so every run is reproducible.

In [ ]:
if "python" in TRACKS:
    py_configs = grid(n_aircraft=[50, 100, 200], range_km=[200, 300, 400])
    py_seeds = range(10)
    py_df = sweep(run_py, py_configs, py_seeds)  # one row per job
    display(py_df.head())

In [ ]:
if "python" in TRACKS:
    s = mean_ci(py_df, ["n_aircraft", "range_km"], "frac_connected")
    fig, ax = plt.subplots(figsize=(5, 3.8))
    for n, g in s.groupby("n_aircraft"):
        ax.errorbar(g.range_km, g["mean"], yerr=g.ci95, marker="o", capsize=3, label=f"{n} aircraft")
    ax.set_xlabel("Air-to-air range [km]")
    ax.set_ylabel("Aircraft with a path to ground")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.savefig(FIG / "py_connectivity.pdf", bbox_inches="tight")

## 4. NixNet experiment (NixNet track)
Example: ground station → aircraft relay → ground station (`flake.nix`). Each air link adds a one-way delay `d` (and optional loss),
so the expected RTT is about 4·d. Experiments are named `relay-d<delay>-l<loss>`; each run is stored in `results/nixnet/<name>/<run>/`.
Start Jupyter with `nix develop -c jupyter lab` so that `nix` is available.

In [ ]:
if "nixnet" in TRACKS:
    rows = []
    for exp in [f"relay-d{d}-l{l}" for d in (1, 5, 20) for l in (0, 1)]:
        for run_dir in run_nixnet(exp, runs="1-3"):
            client = run_dir / "gs-west"
            rows.append({"experiment": exp, "run": run_dir.name,
                         **read_ping(client / "ping.txt"), **read_iperf3(client / "iperf3.json")})
    nix_df = pd.DataFrame(rows)
    nix_df[["delay_ms", "loss_pct_cfg"]] = nix_df.experiment.str.extract(r"d(\d+)-l(\d+)").astype(int)
    display(nix_df.head())

In [ ]:
if "nixnet" in TRACKS:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for loss, g in nix_df.groupby("loss_pct_cfg"):
        s = mean_ci(g, "delay_ms", "rtt_avg_ms")
        axes[0].errorbar(s.delay_ms, s["mean"], yerr=s.ci95, marker="o", capsize=3, label=f"loss {loss}%")
        s = mean_ci(g, "delay_ms", "throughput_mbps")
        axes[1].errorbar(s.delay_ms, s["mean"], yerr=s.ci95, marker="o", capsize=3, label=f"loss {loss}%")
    d = np.array(sorted(nix_df.delay_ms.unique()))
    axes[0].plot(d, 4 * d, "k--", lw=1, label="expected 4·d")
    axes[0].set_ylabel("RTT [ms]")
    axes[1].set_ylabel("TCP throughput [Mbit/s]")
    for ax in axes:
        ax.set_xlabel("One-way delay per air link [ms]")
        ax.grid(alpha=0.3)
        ax.legend()
    fig.tight_layout()
    fig.savefig(FIG / "nixnet_rtt_throughput.pdf", bbox_inches="tight")

## 5. Results and discussion
- **Reproduction:** which results of the paper you reproduced, and how closely (numbers, figures side by side).
- **Differences:** where your results differ and why (model simplifications, parameters, randomness).
- **Extension:** what you changed or added, and what you learned from it.
- **Limitations:** what your experiment cannot show.

## 6. Reproducibility
- **Environment:** the table printed in section 0 (engine, image IDs or Nix version, git commit).
- **How to reproduce:** see `README.md`. Results are cached in `results/` with a `.json` file per job describing how it was produced.
- **Checklist before submitting:**
  - [ ] all results produced with `ENGINE="docker"` (Docker track) or via `nix develop` (NixNet track)
  - [ ] `flake.lock` / pinned `requirements*.txt` committed
  - [ ] fresh clone + README + *Restart & Run All* works
  - [ ] git working tree clean (`git_dirty` is `False` above)